In [66]:
import pandas as pd
from datetime import datetime
from sklearn.neighbors import KernelDensity
import numpy as np
import matplotlib.pyplot as plt

In [67]:
# Leer el archivo CSV
df = pd.read_csv('/home/dario/Study/sim/Surf-and-Skate-Prediction-Model-for-2024-Olympics/skate/skateboard.csv')

# Convertir la columna 'date' a datetime especificando el formato
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')

# Filtrar por lugar 'Park' y fechas anteriores a mayo de 2024
df_filtrado = df[(df['place'] == 'Park')] 
# & (df['date'] < '2024-05-01')]

# Crear diccionarios para hombres y mujeres
mens = {}
womens = {}

# Función para agregar datos al diccionario
def agregar_a_diccionario(diccionario, fila):
    nombre_completo = f"{fila['first_name'].lower()} {fila['last_name'].lower()}"
    if nombre_completo not in diccionario:
        diccionario[nombre_completo] = []
    diccionario[nombre_completo].append([fila['total']])

# Separar los datos por sexo y llenar los diccionarios
for index, row in df_filtrado.iterrows():
    if row['sex'] == 'Mens':
        agregar_a_diccionario(mens, row)
    elif row['sex'] == 'Womens':
        agregar_a_diccionario(womens, row)

# Imprimir los diccionarios para verificar
print("Hombres:", mens)
print("Mujeres:", womens)

Hombres: {'jagger eaton': [[78.0], [78.0], [78.0], [78.0], [84.23], [84.23], [82.2], [93.0], [93.0], [93.0], [93.0], [93.0], [84.36], [84.36], [81.49], [91.61], [91.61], [91.61], [88.33], [88.33], [88.33], [88.33], [77.17], [77.17], [89.49], [34.15], [34.15], [34.15], [34.15], [86.51], [86.51], [85.5]], 'yuro nagahara': [[75.33], [75.33], [75.33], [75.33], [62.0], [62.0], [62.0], [62.0], [62.0], [62.0], [81.45], [81.45], [31.66], [31.66], [31.66], [78.68], [78.68], [78.68], [78.68], [82.0], [82.0], [82.0], [82.0], [84.43], [84.43], [88.14], [79.2], [79.2], [79.2], [71.06], [71.06], [71.06], [78.86], [78.86], [79.2], [81.99], [62.6], [62.6], [62.6], [81.1], [81.1]], 'augusto akio': [[74.0], [74.0], [74.0], [74.0], [85.62], [85.62], [88.76], [92.0], [92.0], [92.0], [92.0], [92.0], [83.25], [83.25], [81.72], [73.04], [73.04], [73.04], [84.5], [84.5], [84.5], [84.5], [78.46], [78.46], [87.59], [77.86], [77.86], [77.86], [77.86], [84.48]], 'tate carew': [[72.66], [72.66], [72.66], [72.66], 

In [68]:
def evaluate_point(x,kde):
    punto_eval = np.array([[x]])
    log_densidad_punto = kde.score_samples(punto_eval)
    return np.exp(log_densidad_punto) 

In [69]:
def global_max(kde):
    
    # Puntos donde queremos evaluar la densidad
    X_d = np.linspace(0, 100,1000)[:, np.newaxis]

    # Evaluar la densidad en los puntos X_d
    log_densidad = kde.score_samples(X_d)
    densidad = np.exp(log_densidad)
    return np.max(densidad)

In [70]:
def kde_function(data):
    # Datos de ejemplo
    X = np.array(data)
    # Instanciar el estimador de densidad de kernel con kernel de Epanechnikov
    kde = KernelDensity(kernel='epanechnikov', bandwidth=2.5).fit(X)

    return kde

In [71]:
def auxiliar_v_a():
    return 100 * np.random.uniform(0,1)

In [72]:
def lista_densidad(kde):
    # Puntos donde queremos evaluar la densidad
    X_d = np.linspace(0, 100,1000)[:, np.newaxis]

    # Evaluar la densidad en los puntos X_d
    log_densidad = kde.score_samples(X_d)
    return np.exp(log_densidad)

In [73]:
# Generar muestras usando el método de aceptación-rechazo
def accept_rejection(kde):

    while True:
        # Generar X ~ U(0, 1/c)
        X = auxiliar_v_a()
        # Generar U ~ U(0, 1)
        U = np.random.uniform(0, 1)
        # Verificar condición de aceptación
        if U <= evaluate_point(X,kde)/ global_max(kde):  # g(x) = 1 para U(0, 1)
            return X 

In [74]:
class Atleta:
    def __init__(self, nombre, datos):
        self.nombre = nombre
        self.datos = datos
        self.kde = kde_function(datos)

In [75]:
Mens=[]
Womens=[]
for key,value in mens.items():
    Mens.append(Atleta(key,value))
for key,value in womens.items():
    Womens.append(Atleta(key,value))

In [76]:
def simulate_round(athletes):
    result=[]
    for i in range(len(athletes)):
        max=0
        for j in range(3):
            x=accept_rejection(athletes[i].kde)
            if x>max:
                max=x
        result.append([athletes[i],max])
    return sorted(result, key=lambda x: x[1], reverse=True)


In [77]:
def simulate_competition(athletes):

    preliminar_round= simulate_round(athletes)

    semifinal_round=[]

    for i in range(16):
        semifinal_round.append(preliminar_round[i][0])
    
    semifinal_result = simulate_round(semifinal_round)

    for i in range(16):
        preliminar_round[i]=semifinal_result[i]

    final_round=[]

    for i in range(8):
        final_round.append(preliminar_round[i][0])
    
    final_result=simulate_round(final_round)

    for i in range(8):
        preliminar_round[i]=final_result[i]

    return preliminar_round


In [79]:
x = simulate_competition(Mens)
for i in range(len(x)):
    print(f"{i+1}- {x[i][0].nombre} : {x[i][1]}")


1- tate carew : 93.83501429109761
2- jagger eaton : 91.09240418619116
3- gavin bottger : 90.7166660519632
4- augusto santos : 87.67645619125072
5- augusto akio : 86.88032102329942
6- pedro barros : 86.07553398069305
7- luiz mariano : 83.74635652655425
8- tom schaar : 76.79736045845823
9- kieran woolley : 81.61194538605157
10- keegan palmer : 81.10956409242065
11- alex sorgente : 79.46406812891534
12- pedro carvalho : 76.59757144160962
13- yuro nagahara : 75.62716662149329
14- alessandro mazzara : 75.29916498847223
15- liam pace : 74.0534746017571
16- keefer wilson : 66.215803451256
17- steven pineiro : 79.95079357081474
18- pedro quintas : 79.09829724767555
19- hericles fagundes : 78.31484465073486
20- luigi cini : 78.14749925002093
21- danny leon : 77.75093815537458
22- heimana reynolds : 76.43432094511358
23- hampus winberg : 76.18357598675303
24- kalani konig : 76.1087760695157
25- viktor solmunde : 74.23839151277633
26- tommy calvert : 73.73258965567916
27- taylor nye : 73.35395228